# ks_perturbed_solutions.ipynb --- example code for simulating and reduced-order modeling the Kuramoto-Sivashinsky equation on the spectral-based grid for a group of solutions perturbed from the base traveling-wave solution.

# Step 1. Build the grid and KS object

In [2]:
import os
import shutil
import pickle

import numpy as np
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

import configs
from SROpInf.grids.grid1d import Grid1DUniformSpectral
from SROpInf.models.ks import KuramotoSivashinsky
from SROpInf.dataloader import FOMDataloader

from plot import plot_results, plot_cumulative_rRMSE_band_t

params = configs.load_configs()
assert params.base_path.rstrip("/").endswith("ks_perturbed_solutions"), \
    f"Expected configs to point at output/ks_perturbed_solutions, got {params.base_path}"
assert params.type_traj_training == "perturbed_solutions", \
    f"Expected type_traj_training='perturbed_solutions', got {params.type_traj_training!r}"
dataloader = FOMDataloader(params)
grid = Grid1DUniformSpectral(params.Lx, params.nx)
fom  = KuramotoSivashinsky(grid, params.nu)
stepper = fom.get_stepper(method="rk3cn", dt=params.dt)

# Step 2. Create multiple initial conditions perturbed from the base profile and collect training and data

In [ ]:

if os.path.exists(params.fig_path_fom):
    shutil.rmtree(params.fig_path_fom)
    os.makedirs(params.fig_path_fom)
if os.path.exists(params.traj_path_fom):
    shutil.rmtree(params.traj_path_fom)
    os.makedirs(params.traj_path_fom)

# 1. Specify a benchmark initial condition:
x = grid.x

# Load the precomputed burn-in IC (state already on the KS attractor). If it is missing,
# abort and instruct the user to run ks_base_solution.ipynb first to generate it.
save_base_ic = False
try:
    q = np.load(params.data_path + "traj_init_base.npy")
except FileNotFoundError:
    raise FileNotFoundError("No pre-computed base initial field found." \
        "Please run the ks_base_solution.ipynb notebook first to generate the base IC on the attractor, \
        then run this notebook again to load it.")

Q = np.zeros((params.nx, len(params.tsave)))

pbar = tqdm(range(len(params.time)), desc="KS FOM")
for idx_time in pbar:
    if idx_time % params.nsave == 0:
        Q[:, idx_time // params.nsave] = q
        pbar.set_postfix(t=f"{idx_time*params.dt:.1f}", norm=f"{grid.norm(q):.2f}")
    q = stepper.step(q)

if save_base_ic:
    np.save(params.data_path + "traj_init_base.npy", q)

# Perceptually-uniform diverging colormap (Cynthia Brewer's RdBu, reversed) centered at u=0;
# filled contours (41 levels) of the chaotic field.
vlim = float(np.ceil(np.max(np.abs(Q))))
fig, ax = plt.subplots(figsize=(10, 10))
pm = ax.contourf(
    grid.x, params.tsave, Q.T,
    levels=np.linspace(-vlim, vlim, 41),
    cmap="RdBu_r", vmin=-vlim, vmax=vlim)
ax.set_xlabel(r"$x$", fontsize=20)
ax.set_ylabel(r"$t$", fontsize=20)
ax.set_title(r"Kuramoto--Sivashinsky $u(x, t)$", fontsize=18)
ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
ax.set_xticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
ax.set_xlim(0, grid.Lx)
ax.tick_params(labelsize=16)
cbar = fig.colorbar(pm, ax=ax, ticks=np.arange(-vlim, vlim + 1, 1))
cbar.set_label(r"$u$", fontsize=16)
cbar.ax.tick_params(labelsize=16)
plt.show()
plt.savefig(params.fig_path_fom + "traj_base.png", dpi=300)
plt.close()

# Based on the base IC associated with the traveling wave, create 10 perturbed ICs by adding small random harmonics.
q_base = np.load(params.data_path + "traj_init_base.npy")
q_base_norm = grid.norm(q_base)
k_min = 2
k_max = int(np.floor(np.sqrt(1 / params.nu))) # ranges from k = 2 to k_max where k_max is the max wavenumber keeping (1 - nu * k_max^2) > 0, i.e. linearly unstable modes

q_list = []
    
for idx_traj in range(params.num_traj_training):
    print(f"Generating traj {idx_traj:3d} via perturbing the benchmark IC with random harmonics...")
    rng = np.random.default_rng(params.random_seed_training + idx_traj)
    perturbation = np.zeros(params.nx)
    for k in range(k_min, k_max + 1):
        a, b = rng.standard_normal(2)
        perturbation += a * np.cos(2 * np.pi * k * grid.x / grid.Lx) + b * np.sin(2 * np.pi * k * grid.x / grid.Lx)
    perturbation /= grid.norm(perturbation)
    q_list.append(q_base + params.training_perturbation_to_benchmark_ratio * q_base_norm * perturbation)
    
q_list = np.stack(q_list, axis=1)  # shape: (nx, num_traj_training)
for i in range(params.num_traj_training):
    np.save(params.fname_traj_init % i, q_list[:, i])

# Visualize the 10 ICs (all on top of each other for the benchmark trace).
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(grid.x, q_base, "k-", lw=2, label="base IC")
for idx_traj in range(params.num_traj_training):
    ax.plot(grid.x, q_list[:, idx_traj], "-", lw=0.8, alpha=0.7, label=f"traj {idx_traj:3d} perturbed IC")
ax.set_xlabel("$x$"); ax.set_ylabel("$q_0$")
ax.set_title(f"{params.num_traj_training} training ICs = base + harmonic perturbation (energy ratio = {params.training_perturbation_to_benchmark_ratio*100:.0f}%)")
ax.legend(loc="upper right", fontsize=8)
plt.show()
plt.savefig(params.fig_path_fom + "traj_init.png", dpi=300)
plt.close()

# Simulate 10 trajectories from the 10 perturbed ICs and visualize them together with the benchmark trajectory.
for idx_traj in range(params.num_traj_training):
    print(f"Simulating traj {idx_traj:3d} from perturbed IC...")
    q = q_list[:, idx_traj]
    Q = np.zeros((params.nx, len(params.tsave)))
    pbar = tqdm(range(len(params.time)), desc=f"KS FOM traj {idx_traj:3d}")
    for idx_time in pbar:
        if idx_time % params.nsave == 0:
            Q[:, idx_time // params.nsave] = q
            pbar.set_postfix(t=f"{idx_time*params.dt:.1f}", norm=f"{grid.norm(q):.2f}")
        q = stepper.step(q)
        
    np.save(params.fname_time, params.tsave)
    np.save(params.fname_traj % ("fom", idx_traj), Q)
    np.save(params.fname_traj_scaled % ("fom", idx_traj), grid.apply_sqrt_inner_product_mass(Q))
    
    vlim = float(np.ceil(np.max(np.abs(Q))))
    fig, ax = plt.subplots(figsize=(10, 10))
    pm = ax.contourf(
        grid.x, params.tsave, Q.T,
        levels=np.linspace(-vlim, vlim, 41),
        cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    ax.set_xlabel(r"$x$", fontsize=20)
    ax.set_ylabel(r"$t$", fontsize=20)
    ax.set_title(f"KS FOM traj {idx_traj:3d} from perturbed IC", fontsize=18)
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
    ax.set_xticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
    ax.set_xlim(0, grid.Lx)
    ax.tick_params(labelsize=16)
    cbar = fig.colorbar(pm, ax=ax, ticks=np.arange(-vlim, vlim + 1, 1))
    cbar.set_label(r"$q$", fontsize=16)
    cbar.ax.tick_params(labelsize=16)
    plt.show()
    plt.savefig(params.fig_path_fom + "traj_%03d.png" % idx_traj, dpi=300)
    plt.close()
    
# Repeat the same procedure for the TESTING dataset. random_seed_testing is disjoint from
# random_seed_training, so these perturbed ICs are guaranteed different from every training
# IC -- a held-out set never seen in training. (Run this cell right after the Step 2 cell:
# it reuses q_base, q_base_norm, k_min, k_max, and q_list from above.)

if os.path.exists(params.fig_path_fom_testing):
    shutil.rmtree(params.fig_path_fom_testing)
    os.makedirs(params.fig_path_fom_testing)
if os.path.exists(params.traj_path_fom_testing):
    shutil.rmtree(params.traj_path_fom_testing)
    os.makedirs(params.traj_path_fom_testing)

q_list_testing = []
for idx_traj in range(params.num_traj_testing):
    print(f"Generating testing traj {idx_traj:3d} via perturbing the benchmark IC with random harmonics...")
    rng = np.random.default_rng(params.random_seed_testing + idx_traj)
    perturbation = np.zeros(params.nx)
    for k in range(k_min, k_max + 1):
        a, b = rng.standard_normal(2)
        perturbation += a * np.cos(2 * np.pi * k * grid.x / grid.Lx) + b * np.sin(2 * np.pi * k * grid.x / grid.Lx)
    perturbation /= grid.norm(perturbation)
    q_list_testing.append(q_base + params.testing_perturbation_to_benchmark_ratio * q_base_norm * perturbation)

q_list_testing = np.stack(q_list_testing, axis=1)  # shape: (nx, num_traj_testing)
for i in range(params.num_traj_testing):
    np.save(params.fname_traj_init_testing % i, q_list_testing[:, i])

# Confirm every testing IC is distinct from all training ICs.
for j in range(params.num_traj_testing):
    dmin = min(grid.norm(q_list_testing[:, j] - q_list[:, i]) for i in range(params.num_traj_training))
    print(f"  testing traj {j:3d}   min distance to any training IC = {dmin:.3e}")

# Simulate the testing trajectories from the perturbed ICs and save under the *_testing file names.
for idx_traj in range(params.num_traj_testing):
    print(f"Simulating testing traj {idx_traj:3d} from perturbed IC...")
    q = q_list_testing[:, idx_traj]
    Q = np.zeros((params.nx, len(params.tsave)))
    pbar = tqdm(range(len(params.time)), desc=f"KS FOM testing traj {idx_traj:3d}")
    for idx_time in pbar:
        if idx_time % params.nsave == 0:
            Q[:, idx_time // params.nsave] = q
            pbar.set_postfix(t=f"{idx_time*params.dt:.1f}", norm=f"{grid.norm(q):.2f}")
        q = stepper.step(q)

    np.save(params.fname_traj % ("fom_testing", idx_traj), Q)
    np.save(params.fname_traj_scaled % ("fom_testing", idx_traj), grid.apply_sqrt_inner_product_mass(Q))

    vlim = float(np.ceil(np.max(np.abs(Q))))
    fig, ax = plt.subplots(figsize=(10, 10))
    pm = ax.contourf(
        grid.x, params.tsave, Q.T,
        levels=np.linspace(-vlim, vlim, 41),
        cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    ax.set_xlabel(r"$x$", fontsize=20)
    ax.set_ylabel(r"$t$", fontsize=20)
    ax.set_title(f"KS FOM testing traj {idx_traj:3d} from perturbed IC", fontsize=18)
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
    ax.set_xticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
    ax.set_xlim(0, grid.Lx)
    ax.tick_params(labelsize=16)
    cbar = fig.colorbar(pm, ax=ax, ticks=np.arange(-vlim, vlim + 1, 1))
    cbar.set_label(r"$q$", fontsize=16)
    cbar.ax.tick_params(labelsize=16)
    plt.show()
    plt.savefig(params.fig_path_fom_testing + "traj_%03d.png" % idx_traj, dpi=300)
    plt.close()

# Step 3. Perform template fitting and symmetry-reduced POD

In [ ]:

from SROpInf.sr_tools import template_fitting
from SROpInf.mode_decomposition import pod
from SROpInf.models.model import SymmetryReducedScaledFullOrderModel

# Assign the template to be q_template = cos(2πx/L) for global, traveling wave
q_template = np.cos(2 * np.pi * grid.x / grid.Lx)
q_template_perp = np.sin(2 * np.pi * grid.x / grid.Lx)
q_template_scaled = grid.apply_sqrt_inner_product_mass(q_template)
q_template_perp_scaled = grid.apply_sqrt_inner_product_mass(q_template_perp)
q_template_dx_scaled = grid.diff_x(q_template_scaled)
q_template_dxx_scaled = grid.diff_x(q_template_scaled, order=2)

fom_sr = SymmetryReducedScaledFullOrderModel(
    grid = grid,
    base_fom = fom,
    q_template_dx_scaled = q_template_dx_scaled,
    q_template_dxx_scaled = q_template_dxx_scaled)

np.save(params.data_path + "q_template.npy", q_template)
np.save(params.data_path + "q_template_perp.npy", q_template_perp)
np.save(params.data_path + "q_template_scaled.npy", q_template_scaled)
np.save(params.data_path + "q_template_perp_scaled.npy", q_template_perp_scaled)

Q_fitted_scaled_all = np.zeros((q_template.shape[0], len(params.tsave), params.num_traj_training))

for idx_traj in range(params.num_traj_training):
    print(f"Performing template fitting for traj {idx_traj:2d}...")
    Q = dataloader.get_traj(which_traj = idx_traj, which_snapshot = range(len(params.tsave)))
    Q_fitted, c = template_fitting(Q = Q, grid = grid, q_template = q_template, q_template_perp = q_template_perp)
    Q_fitted_scaled = grid.apply_sqrt_inner_product_mass(Q_fitted)
    
    inv_shift_speed_denom = fom_sr.inv_shift_speed_denom(Q_fitted_scaled)
    
    np.save(params.fname_traj_fitted % ("fom", idx_traj), Q_fitted)
    np.save(params.fname_traj_fitted_scaled % ("fom", idx_traj), Q_fitted_scaled)
    np.save(params.fname_traj_init_fitted_scaled % idx_traj, Q_fitted_scaled[:, 0])
    
    # also save right-hand-side terms for operator inference
    np.save(params.fname_rhs_fitted_scaled % ("fom", idx_traj), fom.rhs_scaled(Q_fitted_scaled))
    np.save(params.fname_shift_amount % ("fom", idx_traj), c)
    np.save(params.fname_inv_shift_speed_denom % ("fom", idx_traj), inv_shift_speed_denom)
    
    Q_fitted_scaled_all[:, :, idx_traj] = Q_fitted_scaled
    
    # Diagnostic: plot the inverse shift-speed denominator 1/D vs t. fom_sr returns the
    # unregularized 1/D; a spike (D -> 0) flags a near-singular shift-speed event. The ROM
    # class regularizes it as D/(D^2 + tau^2), but the FOM quantity plotted here is plain 1/D.
    tau = params.shift_speed_denom_threshold
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(params.tsave, inv_shift_speed_denom, lw=1, label=r"$1/D$")
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"inv. shift-speed denom.")
    ax.set_title(f"KS FOM traj {idx_traj:2d}: inv_shift_speed_denom (τ={tau:.1e})")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.savefig(params.fig_path_fom + "inv_shift_speed_denom_%03d.png" % idx_traj, dpi=300)
    plt.close()
    
    vlim = float(np.ceil(np.max(np.abs(Q_fitted))))
    fig, ax = plt.subplots(figsize=(10, 10))
    pm = ax.contourf(
        grid.x, params.tsave, Q_fitted.T,
        levels=np.linspace(-vlim, vlim, 41),
        cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    ax.set_xlabel(r"$x$", fontsize=20)
    ax.set_ylabel(r"$t$", fontsize=20)
    ax.set_title(f"KS FOM template-fitted traj {idx_traj:2d} from perturbed IC", fontsize=18)
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
    ax.set_xticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
    ax.set_xlim(0, grid.Lx)
    ax.tick_params(labelsize=16)
    cbar = fig.colorbar(pm, ax=ax, ticks=np.arange(-vlim, vlim + 1, 1))
    cbar.set_label(r"$q$", fontsize=16)
    cbar.ax.tick_params(labelsize=16)
    plt.savefig(params.fig_path_fom + "traj_fitted_%03d.png" % idx_traj, dpi=300)
    plt.close()

for idx_traj in range(params.num_traj_testing):
    print(f"Performing template fitting for testing traj {idx_traj:2d}...")
    Q = np.load(params.fname_traj % ("fom_testing", idx_traj))
    Q_fitted, c = template_fitting(Q = Q, grid = grid, q_template = q_template, q_template_perp = q_template_perp)
    Q_fitted_scaled = grid.apply_sqrt_inner_product_mass(Q_fitted)
    
    inv_shift_speed_denom = fom_sr.inv_shift_speed_denom(Q_fitted_scaled)
    
    np.save(params.fname_traj_fitted % ("fom_testing", idx_traj), Q_fitted)
    np.save(params.fname_traj_fitted_scaled % ("fom_testing", idx_traj), Q_fitted_scaled)
    np.save(params.fname_traj_init_testing_fitted_scaled % idx_traj, Q_fitted_scaled[:, 0])
    
    # also save right-hand-side terms for operator inference
    np.save(params.fname_shift_amount % ("fom_testing", idx_traj), c)
    np.save(params.fname_inv_shift_speed_denom % ("fom_testing", idx_traj), inv_shift_speed_denom)
    
    # Diagnostic: plot the inverse shift-speed denominator 1/D vs t. fom_sr returns the
    # unregularized 1/D; a spike (D -> 0) flags a near-singular shift-speed event. The ROM
    # class regularizes it as D/(D^2 + tau^2), but the FOM quantity plotted here is plain 1/D.
    tau = params.shift_speed_denom_threshold
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(params.tsave, inv_shift_speed_denom, lw=1, label=r"$1/D$")
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"inv. shift-speed denom.")
    ax.set_title(f"KS FOM testing traj {idx_traj:2d}: inv_shift_speed_denom (τ={tau:.1e})")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.savefig(params.fig_path_fom_testing + "inv_shift_speed_denom_%03d.png" % idx_traj, dpi=300)
    plt.close()
    
    vlim = float(np.ceil(np.max(np.abs(Q_fitted))))
    fig, ax = plt.subplots(figsize=(10, 10))
    pm = ax.contourf(
        grid.x, params.tsave, Q_fitted.T,
        levels=np.linspace(-vlim, vlim, 41),
        cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    ax.set_xlabel(r"$x$", fontsize=20)
    ax.set_ylabel(r"$t$", fontsize=20)
    ax.set_title(f"KS FOM template-fitted testing traj {idx_traj:2d} from perturbed IC", fontsize=18)
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
    ax.set_xticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
    ax.set_xlim(0, grid.Lx)
    ax.tick_params(labelsize=16)
    cbar = fig.colorbar(pm, ax=ax, ticks=np.arange(-vlim, vlim + 1, 1))
    cbar.set_label(r"$q$", fontsize=16)
    cbar.ax.tick_params(labelsize=16)
    plt.savefig(params.fig_path_fom_testing + "traj_fitted_%03d.png" % idx_traj, dpi=300)
    plt.close()    
    
phi_scaled, r = pod(Q_scaled = Q_fitted_scaled_all.reshape(Q_fitted_scaled_all.shape[0], -1),
                    r = params.r,
                    r_tentative = params.num_traj_training * 16,
                    cumulative_energy_ratio=params.pod_energy_threshold)

np.save(params.data_path + "phi_srpod.npy", phi_scaled)

# Step 4. S-R POD-Galerkin ROM via Galerkin projection onto the S-R POD subspace
# (10 perturbed solutions, r = 20 modes)

In [ ]:

from SROpInf.models.model import SymmetryReducedScaledFullOrderModel

phi_scaled = np.load(params.data_path + "phi_srpod.npy")
q_template_scaled = np.load(params.data_path + "q_template_scaled.npy")
q_template_dx_scaled = grid.diff_x(q_template_scaled)
q_template_dxx_scaled = grid.diff_x(q_template_dx_scaled)

if os.path.exists(params.fig_path_srpodgal_spec):
    shutil.rmtree(params.fig_path_srpodgal_spec)
    os.makedirs(params.fig_path_srpodgal_spec)
if os.path.exists(params.traj_path_srpodgal_spec):
    shutil.rmtree(params.traj_path_srpodgal_spec)
    os.makedirs(params.traj_path_srpodgal_spec)

c_fom = '#000000'
c_srpodgal = '#6A3D9A'
l_fom = '-'
l_srpodgal = '-'

model_path_list = {"traj_scaled": params.fname_traj_scaled, 
                   "error": params.fname_error,
                   "shift_amount": params.fname_shift_amount,
                   "inv_shift_speed_denom": params.fname_inv_shift_speed_denom}
                   
model_name_list = ["FOM", "S-R Galerkin ROM"]
model_abbrev_list = ["fom", "srpodgal_spectral"]
color_list = [c_fom, c_srpodgal]
linestyle_list = [l_fom, l_srpodgal]

plot_config_list = {
    "model_name_list": model_name_list,
    "model_abbrev_list": model_abbrev_list,
    "color_list": color_list,
    "linestyle_list": linestyle_list
}

fom_sr = SymmetryReducedScaledFullOrderModel(
    grid = grid,
    base_fom = fom,
    q_template_dx_scaled = q_template_dx_scaled,
    q_template_dxx_scaled = q_template_dxx_scaled)

rom_srpodgal = fom_sr.project(
    poly_comp = params.poly_comp,
    phi_scaled = phi_scaled,
    shift_speed_denom_threshold = params.shift_speed_denom_threshold
)

for idx_traj in range(params.num_traj_training):
    print(f"Simulating {model_name_list[-1]} traj {idx_traj:2d}...")
    q0_fitted_scaled = np.load(params.fname_traj_init_fitted_scaled % idx_traj)
    c0 = np.load(params.fname_shift_amount % ("fom", idx_traj))[0]
    z0 = rom_srpodgal.full_to_latent(q0_fitted_scaled)
    output_list = rom_srpodgal.sample_and_compare(idx_traj, y0 = np.hstack([z0, c0]), t_eval = params.tsave,
                        model_list = model_abbrev_list,
                        model_path = model_path_list)
    
    plot_results(
        output_list = output_list,
        params = params,
        grid = grid,
        idx_traj = idx_traj,
        fig_path = params.fig_path_srpodgal_spec,
        config_list = plot_config_list,
        scale_func = grid.apply_sqrt_inner_product_mass,
    )

plot_cumulative_rRMSE_band_t(
    model_name_list   = model_name_list[1:],
    model_abbrev_list = model_abbrev_list[1:],
    color_list        = color_list[1:],
    num_traj          = params.num_traj_training,
    tsave             = params.tsave,
    fname_traj_scaled = params.fname_traj_scaled,
    fom_abbrev        = "fom",
    fig_path          = params.fig_path_srpodgal_spec,
    dataset_label     = "training",
)        
    

# Step 5. S-R OpInf ROM with the optimal penalty weights (1e-13, 1e-7)

In [3]:

from SROpInf.sr_tools import sropinf
from SROpInf.models.model import SymmetryReducedScaledFullOrderModel, SymmetryReducedScaledReducedOrderModel

phi_scaled = np.load(params.data_path + "phi_srpod.npy")
q_template_scaled = np.load(params.data_path + "q_template_scaled.npy")
q_template_dx_scaled = grid.diff_x(q_template_scaled)
q_template_dxx_scaled = grid.diff_x(q_template_dx_scaled)

if os.path.exists(params.fig_path_sropinf_spec_reg):
    shutil.rmtree(params.fig_path_sropinf_spec_reg)
    os.makedirs(params.fig_path_sropinf_spec_reg)
if os.path.exists(params.traj_path_sropinf_spec_reg):
    shutil.rmtree(params.traj_path_sropinf_spec_reg)
    os.makedirs(params.traj_path_sropinf_spec_reg)
    
c_fom = '#000000'
c_srpodgal = '#6A3D9A'
c_sropinf_reg = "#FF3C00"
l_fom = '-'
l_srpodgal = '-'
l_sropinf_reg = '-'

model_path_list = {"traj_scaled": params.fname_traj_scaled, 
                   "error": params.fname_error,
                   "shift_amount": params.fname_shift_amount,
                   "inv_shift_speed_denom": params.fname_inv_shift_speed_denom}
                   
model_name_list = ["FOM", "S-R Galerkin ROM", "S-R OpInf ROM (optimal penalty weights)"]
model_abbrev_list = ["fom", "srpodgal_spectral", "sropinf_spectral_regularized"]
color_list = [c_fom, c_srpodgal, c_sropinf_reg]
linestyle_list = [l_fom, l_srpodgal, l_sropinf_reg]

plot_config_list = {
    "model_name_list": model_name_list,
    "model_abbrev_list": model_abbrev_list,
    "color_list": color_list,
    "linestyle_list": linestyle_list
}

fom_sr = SymmetryReducedScaledFullOrderModel(
    grid = grid,
    base_fom = fom,
    q_template_dx_scaled = q_template_dx_scaled,
    q_template_dxx_scaled = q_template_dxx_scaled)

# 1. Load the dataset for training of S-R OpInf
Q_fitted_scaled = dataloader.get_traj(
    which_traj = range(params.num_traj_training),
    which_snapshot = range(len(params.tsave)),
    is_scaled = True,
    is_template_fitted = True
)

c = dataloader.get_shift_amount(
    which_traj = range(params.num_traj_training),
    which_snapshot = range(len(params.tsave))
)

tensors_sropinf_reg = sropinf(
    poly_comp = params.poly_comp,
    Phi = phi_scaled,
    Q_fitted = Q_fitted_scaled,
    fom_sr = fom_sr,
    re_projection_option = False,
    grid_search_option = False,
    # grid_search_option = True,
    cross_validation_option = False,
    # cross_validation_option = True,
    penalty_weight_rhs_poly = params.opinf_penalty_weight_rhs_poly,
    penalty_weight_dcdt_numer = params.opinf_penalty_weight_dcdt_numer,
    t_eval = params.tsave,
    shift_amount = c,
    fig_path = params.fig_path_sropinf_spec_reg,
)

rom_sropinf_reg = SymmetryReducedScaledReducedOrderModel.build(
    poly_comp = params.poly_comp,
    phi_scaled = phi_scaled,
    tensors = tensors_sropinf_reg,
    q_template_dxx_scaled = q_template_dxx_scaled,
    shift_func = fom_sr.shift_x,
    shift_speed_denom_threshold = params.shift_speed_denom_threshold
)

for idx_traj in range(params.num_traj_training):
    print(f"Simulating {model_name_list[-1]} traj {idx_traj:2d}...")
    q0_fitted_scaled = np.load(params.fname_traj_init_fitted_scaled % idx_traj)
    c0 = np.load(params.fname_shift_amount % ("fom", idx_traj))[0]
    z0 = rom_sropinf_reg.full_to_latent(q0_fitted_scaled)
    output_list = rom_sropinf_reg.sample_and_compare(idx_traj, y0 = np.hstack([z0, c0]), t_eval = params.tsave,
                        model_list = model_abbrev_list,
                        model_path = model_path_list)
    
    plot_results(
        output_list = output_list,
        params = params,
        grid = grid,
        idx_traj = idx_traj,
        fig_path = params.fig_path_sropinf_spec_reg,
        config_list = plot_config_list,
        scale_func = grid.apply_sqrt_inner_product_mass,
    )
    
with open(params.data_path + "tensors_sropinf_spectral_regularized.pkl", "wb") as f:
    pickle.dump(tensors_sropinf_reg, f)
    
plot_cumulative_rRMSE_band_t(
    model_name_list   = model_name_list[1:],
    model_abbrev_list = model_abbrev_list[1:],
    color_list        = color_list[1:],
    num_traj          = params.num_traj_training,
    tsave             = params.tsave,
    fname_traj_scaled = params.fname_traj_scaled,
    fom_abbrev        = "fom",
    fig_path          = params.fig_path_sropinf_spec_reg,
    dataset_label     = "training",
)    

Simulating S-R OpInf ROM (optimal penalty weights) traj  0...
Traj 0: relative RMSE 0.08%
Simulating S-R OpInf ROM (optimal penalty weights) traj  1...
Traj 1: relative RMSE 0.70%
Simulating S-R OpInf ROM (optimal penalty weights) traj  2...
Traj 2: relative RMSE 0.17%
Simulating S-R OpInf ROM (optimal penalty weights) traj  3...
Traj 3: relative RMSE 0.11%
Simulating S-R OpInf ROM (optimal penalty weights) traj  4...
Traj 4: relative RMSE 0.21%
Simulating S-R OpInf ROM (optimal penalty weights) traj  5...
Traj 5: relative RMSE 0.04%
Simulating S-R OpInf ROM (optimal penalty weights) traj  6...
Traj 6: relative RMSE 0.06%
Simulating S-R OpInf ROM (optimal penalty weights) traj  7...
Traj 7: relative RMSE 0.28%
Simulating S-R OpInf ROM (optimal penalty weights) traj  8...
Traj 8: relative RMSE 0.37%
Simulating S-R OpInf ROM (optimal penalty weights) traj  9...
Traj 9: relative RMSE 0.27%


# Step 6. S-R POD-Galerkin ROM applied for reconstruction of testing trajectories

In [4]:
from SROpInf.models.model import SymmetryReducedScaledFullOrderModel

phi_scaled = np.load(params.data_path + "phi_srpod.npy")
q_template_scaled = np.load(params.data_path + "q_template_scaled.npy")
q_template_dx_scaled = grid.diff_x(q_template_scaled)
q_template_dxx_scaled = grid.diff_x(q_template_dx_scaled)

c_fom = '#000000'
c_srpodgal = '#6A3D9A'
l_fom = '-'
l_srpodgal = '-'

if os.path.exists(params.fig_path_srpodgal_spec_testing):
    shutil.rmtree(params.fig_path_srpodgal_spec_testing)
    os.makedirs(params.fig_path_srpodgal_spec_testing)
if os.path.exists(params.traj_path_srpodgal_spec_testing):
    shutil.rmtree(params.traj_path_srpodgal_spec_testing)
    os.makedirs(params.traj_path_srpodgal_spec_testing)

model_path_list = {"traj_scaled": params.fname_traj_scaled, 
                   "error": params.fname_error,
                   "shift_amount": params.fname_shift_amount,
                   "inv_shift_speed_denom": params.fname_inv_shift_speed_denom}
                   
model_name_list = ["FOM", "S-R Galerkin ROM"]
model_abbrev_list = ["fom_testing", "srpodgal_spectral_testing"]
color_list = [c_fom, c_srpodgal]
linestyle_list = [l_fom, l_srpodgal]

plot_config_list = {
    "model_name_list": model_name_list,
    "model_abbrev_list": model_abbrev_list,
    "color_list": color_list,
    "linestyle_list": linestyle_list
}

fom_sr = SymmetryReducedScaledFullOrderModel(
    grid = grid,
    base_fom = fom,
    q_template_dx_scaled = q_template_dx_scaled,
    q_template_dxx_scaled = q_template_dxx_scaled)

rom_srpodgal = fom_sr.project(
    poly_comp = params.poly_comp,
    phi_scaled = phi_scaled,
    shift_speed_denom_threshold = params.shift_speed_denom_threshold
)

for idx_traj in range(params.num_traj_testing):
    print(f"Simulating {model_name_list[-1]} traj {idx_traj:2d}...")
    q0_fitted_scaled = np.load(params.fname_traj_init_testing_fitted_scaled % idx_traj)
    c0 = np.load(params.fname_shift_amount % ("fom_testing", idx_traj))[0]
    z0 = rom_srpodgal.full_to_latent(q0_fitted_scaled)
    output_list = rom_srpodgal.sample_and_compare(idx_traj, y0 = np.hstack([z0, c0]), t_eval = params.tsave,
                        model_list = model_abbrev_list,
                        model_path = model_path_list)
    
    plot_results(
        output_list = output_list,
        params = params,
        grid = grid,
        idx_traj = idx_traj,
        fig_path = params.fig_path_srpodgal_spec_testing,
        config_list = plot_config_list,
        scale_func = grid.apply_sqrt_inner_product_mass,
    )

plot_cumulative_rRMSE_band_t(
    model_name_list   = model_name_list[1:],
    model_abbrev_list = model_abbrev_list[1:],
    color_list        = color_list[1:],
    num_traj          = params.num_traj_testing,
    tsave             = params.tsave,
    fname_traj_scaled = params.fname_traj_scaled,
    fom_abbrev        = "fom_testing",
    fig_path          = params.fig_path_srpodgal_spec_testing,
    dataset_label     = "testing",
)

# --- dump per-trajectory total relative RMSE to a txt (perturbation-amplitude sweep) ---
# error_*.npy is already written per trajectory by sample_and_compare; here we also dump a
# human-readable txt tagged with the CURRENT perturbation amplitude so repeated runs at different
# amplitudes accumulate without overwriting. For each amplitude: change
# testing_perturbation_to_benchmark_ratio in configs, then re-run Step 2 -> 3 -> 6 -> 7.
_abbrev = "srpodgal_spectral_testing"
_amp = params.testing_perturbation_to_benchmark_ratio * 100
_errs = [float(np.load(params.fname_error % (_abbrev, _j))) for _j in range(params.num_traj_testing)]
_sweep_dir = params.base_path + "sweep_rRMSE/"
os.makedirs(_sweep_dir, exist_ok=True)
_txt = _sweep_dir + f"{_abbrev}_amp{_amp:g}.txt"
with open(_txt, "w") as _f:
    _f.write(f"# {_abbrev}  testing  perturbation_amplitude={_amp:g}% of base L2 norm  num_traj={params.num_traj_testing}\n")
    _f.write(f"# mean={np.nanmean(_errs):.6f}  std={np.nanstd(_errs):.6f}\n")
    _f.write("# traj_idx\ttotal_relative_RMSE\n")
    for _j, _e in enumerate(_errs):
        _f.write(f"{_j:3d}\t{_e:.6f}\n")
print("wrote", _txt)


Simulating S-R Galerkin ROM traj  0...
Traj 0: relative RMSE 1.17%
Simulating S-R Galerkin ROM traj  1...
Traj 1: relative RMSE 116.68%
Simulating S-R Galerkin ROM traj  2...
Traj 2: relative RMSE 6.28%
Simulating S-R Galerkin ROM traj  3...
Traj 3: relative RMSE 7.58%
Simulating S-R Galerkin ROM traj  4...
Traj 4: relative RMSE 5.46%
Simulating S-R Galerkin ROM traj  5...
Traj 5: relative RMSE 6.17%
Simulating S-R Galerkin ROM traj  6...
Traj 6: relative RMSE 2.57%
Simulating S-R Galerkin ROM traj  7...
Traj 7: relative RMSE 9.68%
Simulating S-R Galerkin ROM traj  8...
Traj 8: relative RMSE 8.50%
Simulating S-R Galerkin ROM traj  9...
Traj 9: relative RMSE 4.56%
Simulating S-R Galerkin ROM traj 10...
Traj 10: relative RMSE 2.65%
Simulating S-R Galerkin ROM traj 11...
Traj 11: relative RMSE 3.92%
Simulating S-R Galerkin ROM traj 12...
Traj 12: relative RMSE 2.45%
Simulating S-R Galerkin ROM traj 13...
Traj 13: relative RMSE 2.22%
Simulating S-R Galerkin ROM traj 14...
Traj 14: relative

# Step 7. S-R POD-OpInf ROM trained with the best regularizer pair applied for reconstruction of testing trajectories

In [5]:
from SROpInf.models.model import SymmetryReducedScaledFullOrderModel, SymmetryReducedScaledReducedOrderModel

phi_scaled = np.load(params.data_path + "phi_srpod.npy")

with open(params.data_path + "tensors_sropinf_spectral_regularized.pkl", "rb") as f:
    tensors_sropinf_reg = pickle.load(f)
    
q_template_scaled = np.load(params.data_path + "q_template_scaled.npy")
q_template_dx_scaled = grid.diff_x(q_template_scaled)
q_template_dxx_scaled = grid.diff_x(q_template_dx_scaled)

c_fom = '#000000'
c_srpodgal = '#6A3D9A'
c_sropinf_reg = "#FF3C00"
l_fom = '-'
l_srpodgal = '-'
l_sropinf_reg = '-'

if os.path.exists(params.fig_path_sropinf_spec_reg_testing):
    shutil.rmtree(params.fig_path_sropinf_spec_reg_testing)
    os.makedirs(params.fig_path_sropinf_spec_reg_testing)
if os.path.exists(params.traj_path_sropinf_spec_reg_testing):
    shutil.rmtree(params.traj_path_sropinf_spec_reg_testing)
    os.makedirs(params.traj_path_sropinf_spec_reg_testing)

model_path_list = {"traj_scaled": params.fname_traj_scaled, 
                   "error": params.fname_error,
                   "shift_amount": params.fname_shift_amount,
                   "inv_shift_speed_denom": params.fname_inv_shift_speed_denom}
                   
model_name_list = ["FOM", "S-R Galerkin ROM", "S-R OpInf ROM (optimal penalty weights)"]
model_abbrev_list = ["fom_testing", "srpodgal_spectral_testing", "sropinf_spectral_regularized_testing"]
color_list = [c_fom, c_srpodgal, c_sropinf_reg]
linestyle_list = [l_fom, l_srpodgal, l_sropinf_reg]

plot_config_list = {
    "model_name_list": model_name_list,
    "model_abbrev_list": model_abbrev_list,
    "color_list": color_list,
    "linestyle_list": linestyle_list
}

fom_sr = SymmetryReducedScaledFullOrderModel(
    grid = grid,
    base_fom = fom,
    q_template_dx_scaled = q_template_dx_scaled,
    q_template_dxx_scaled = q_template_dxx_scaled)

rom_sropinf_reg = SymmetryReducedScaledReducedOrderModel.build(
    poly_comp = params.poly_comp,
    phi_scaled = phi_scaled,
    tensors = tensors_sropinf_reg,
    q_template_dxx_scaled = q_template_dxx_scaled,
    shift_func = fom_sr.shift_x,
    shift_speed_denom_threshold = params.shift_speed_denom_threshold
)

for idx_traj in range(params.num_traj_testing):
    print(f"Simulating {model_name_list[-1]} traj {idx_traj:2d}...")
    q0_fitted_scaled = np.load(params.fname_traj_init_testing_fitted_scaled % idx_traj)
    c0 = np.load(params.fname_shift_amount % ("fom_testing", idx_traj))[0]
    z0 = rom_sropinf_reg.full_to_latent(q0_fitted_scaled)
    output_list = rom_sropinf_reg.sample_and_compare(idx_traj, y0 = np.hstack([z0, c0]), t_eval = params.tsave,
                        model_list = model_abbrev_list,
                        model_path = model_path_list)
    
    plot_results(
        output_list = output_list,
        params = params,
        grid = grid,
        idx_traj = idx_traj,
        fig_path = params.fig_path_sropinf_spec_reg_testing,
        config_list = plot_config_list,
        scale_func = grid.apply_sqrt_inner_product_mass,
    )

plot_cumulative_rRMSE_band_t(
    model_name_list   = model_name_list[1:],
    model_abbrev_list = model_abbrev_list[1:],
    color_list        = color_list[1:],
    num_traj          = params.num_traj_testing,
    tsave             = params.tsave,
    fname_traj_scaled = params.fname_traj_scaled,
    fom_abbrev        = "fom_testing",
    fig_path          = params.fig_path_sropinf_spec_reg_testing,
    dataset_label     = "testing",
)

# --- dump per-trajectory total relative RMSE to a txt (perturbation-amplitude sweep) ---
# error_*.npy is already written per trajectory by sample_and_compare; here we also dump a
# human-readable txt tagged with the CURRENT perturbation amplitude so repeated runs at different
# amplitudes accumulate without overwriting. For each amplitude: change
# testing_perturbation_to_benchmark_ratio in configs, then re-run Step 2 -> 3 -> 6 -> 7.
_abbrev = "sropinf_spectral_regularized_testing"
_amp = params.testing_perturbation_to_benchmark_ratio * 100
_errs = [float(np.load(params.fname_error % (_abbrev, _j))) for _j in range(params.num_traj_testing)]
_sweep_dir = params.base_path + "sweep_rRMSE/"
os.makedirs(_sweep_dir, exist_ok=True)
_txt = _sweep_dir + f"{_abbrev}_amp{_amp:g}.txt"
with open(_txt, "w") as _f:
    _f.write(f"# {_abbrev}  testing  perturbation_amplitude={_amp:g}% of base L2 norm  num_traj={params.num_traj_testing}\n")
    _f.write(f"# mean={np.nanmean(_errs):.6f}  std={np.nanstd(_errs):.6f}\n")
    _f.write("# traj_idx\ttotal_relative_RMSE\n")
    for _j, _e in enumerate(_errs):
        _f.write(f"{_j:3d}\t{_e:.6f}\n")
print("wrote", _txt)


Simulating S-R OpInf ROM (optimal penalty weights) traj  0...
Traj 0: relative RMSE 0.21%
Simulating S-R OpInf ROM (optimal penalty weights) traj  1...
Traj 1: relative RMSE 2.36%
Simulating S-R OpInf ROM (optimal penalty weights) traj  2...
Traj 2: relative RMSE 0.81%
Simulating S-R OpInf ROM (optimal penalty weights) traj  3...
Traj 3: relative RMSE 0.78%
Simulating S-R OpInf ROM (optimal penalty weights) traj  4...
Traj 4: relative RMSE 0.24%
Simulating S-R OpInf ROM (optimal penalty weights) traj  5...
Traj 5: relative RMSE 0.44%
Simulating S-R OpInf ROM (optimal penalty weights) traj  6...
Traj 6: relative RMSE 0.74%
Simulating S-R OpInf ROM (optimal penalty weights) traj  7...
Traj 7: relative RMSE 0.79%
Simulating S-R OpInf ROM (optimal penalty weights) traj  8...
Traj 8: relative RMSE 0.71%
Simulating S-R OpInf ROM (optimal penalty weights) traj  9...
Traj 9: relative RMSE 0.26%
Simulating S-R OpInf ROM (optimal penalty weights) traj 10...
Traj 10: relative RMSE 0.39%
Simulatin

# Step 8. S-R POD-Galerkin ROM with denominator regularization applied for reconstruction of testing trajectories

In [ ]:
from SROpInf.models.model import SymmetryReducedScaledFullOrderModel

phi_scaled = np.load(params.data_path + "phi_srpod.npy")
q_template_scaled = np.load(params.data_path + "q_template_scaled.npy")
q_template_dx_scaled = grid.diff_x(q_template_scaled)
q_template_dxx_scaled = grid.diff_x(q_template_dx_scaled)

c_fom = '#000000'
c_srpodgal = '#6A3D9A'
c_srpodgal_denom_reg = "#FF3C00"
l_fom = '-'
l_srpodgal = '-'
l_srpodgal_denom_reg = '--'

if os.path.exists(params.fig_path_srpodgal_spec_testing_denom_reg):
    shutil.rmtree(params.fig_path_srpodgal_spec_testing_denom_reg)
    os.makedirs(params.fig_path_srpodgal_spec_testing_denom_reg)
if os.path.exists(params.traj_path_srpodgal_spec_testing_denom_reg):
    shutil.rmtree(params.traj_path_srpodgal_spec_testing_denom_reg)
    os.makedirs(params.traj_path_srpodgal_spec_testing_denom_reg)

model_path_list = {"traj_scaled": params.fname_traj_scaled, 
                   "error": params.fname_error,
                   "shift_amount": params.fname_shift_amount,
                   "inv_shift_speed_denom": params.fname_inv_shift_speed_denom}
                   
model_name_list = ["FOM", "S-R Galerkin ROM", "S-R Galerkin ROM (regularized denominator)"]
model_abbrev_list = ["fom_testing", "srpodgal_spectral_testing", "srpodgal_spectral_testing_denominator_regularized"]
color_list = [c_fom, c_srpodgal, c_srpodgal_denom_reg]
linestyle_list = [l_fom, l_srpodgal, l_srpodgal_denom_reg]

plot_config_list = {
    "model_name_list": model_name_list,
    "model_abbrev_list": model_abbrev_list,
    "color_list": color_list,
    "linestyle_list": linestyle_list
}

fom_sr = SymmetryReducedScaledFullOrderModel(
    grid = grid,
    base_fom = fom,
    q_template_dx_scaled = q_template_dx_scaled,
    q_template_dxx_scaled = q_template_dxx_scaled)

rom_srpodgal = fom_sr.project(
    poly_comp = params.poly_comp,
    phi_scaled = phi_scaled,
    shift_speed_denom_threshold = params.shift_speed_denom_threshold
)

for idx_traj in range(params.num_traj_testing):
    print(f"Simulating {model_name_list[-1]} traj {idx_traj:2d}...")
    q0_fitted_scaled = np.load(params.fname_traj_init_testing_fitted_scaled % idx_traj)
    c0 = np.load(params.fname_shift_amount % ("fom_testing", idx_traj))[0]
    z0 = rom_srpodgal.full_to_latent(q0_fitted_scaled)
    output_list = rom_srpodgal.sample_and_compare(idx_traj, y0 = np.hstack([z0, c0]), t_eval = params.tsave,
                        model_list = model_abbrev_list,
                        model_path = model_path_list)
    
    plot_results(
        output_list = output_list,
        params = params,
        grid = grid,
        idx_traj = idx_traj,
        fig_path = params.fig_path_srpodgal_spec_testing_denom_reg,
        config_list = plot_config_list,
        scale_func = grid.apply_sqrt_inner_product_mass,
    )

plot_cumulative_rRMSE_band_t(
    model_name_list   = model_name_list[1:],
    model_abbrev_list = model_abbrev_list[1:],
    color_list        = color_list[1:],
    num_traj          = params.num_traj_testing,
    tsave             = params.tsave,
    fname_traj_scaled = params.fname_traj_scaled,
    fom_abbrev        = "fom_testing",
    fig_path          = params.fig_path_srpodgal_spec_testing_denom_reg,
    dataset_label     = "testing",
)

# --- dump per-trajectory total relative RMSE to a txt (perturbation-amplitude sweep) ---
# error_*.npy is already written per trajectory by sample_and_compare; here we also dump a
# human-readable txt tagged with the CURRENT perturbation amplitude so repeated runs at different
# amplitudes accumulate without overwriting. For each amplitude: change
# testing_perturbation_to_benchmark_ratio in configs, then re-run Step 2 -> 3 -> 6 -> 7.
_abbrev = "srpodgal_spectral_testing_denominator_regularized"
_amp = params.testing_perturbation_to_benchmark_ratio * 100
_errs = [float(np.load(params.fname_error % (_abbrev, _j))) for _j in range(params.num_traj_testing)]
_sweep_dir = params.base_path + "sweep_rRMSE/"
os.makedirs(_sweep_dir, exist_ok=True)
_txt = _sweep_dir + f"{_abbrev}_amp{_amp:g}.txt"
with open(_txt, "w") as _f:
    _f.write(f"# {_abbrev}  testing  perturbation_amplitude={_amp:g}% of base L2 norm  num_traj={params.num_traj_testing}\n")
    _f.write(f"# mean={np.nanmean(_errs):.6f}  std={np.nanstd(_errs):.6f}\n")
    _f.write("# traj_idx\ttotal_relative_RMSE\n")
    for _j, _e in enumerate(_errs):
        _f.write(f"{_j:3d}\t{_e:.6f}\n")
print("wrote", _txt)


# Step 9. Load prediction errors of testing dataset generated from perturbed base solution with various perturbation amplitudes. Plot the change in mean and variance of errors across trajectories with different perturbation amplitudes. 

In [ ]:
# === Fig 5 aggregator: box plots of testing error vs perturbation amplitude (log y) ===
# Reads every {abbrev}_amp*.txt from sweep_rRMSE/ (READ-ONLY; never writes/deletes there) and draws,
# at each amplitude, a grouped box plot of the per-trajectory testing relative RMSE for each ROM.
# Thin boxes + categorical spacing + alternating background shading so each amplitude's pair reads
# as one group; log y-axis. Diverged trajectories are excluded from the box and annotated "N div".
# All translucency is pre-blended into solid colors via plot._on_white -- the EPS backend renders
# alpha-carrying artists as fully opaque, so genuine alpha must never reach savefig(.eps).
import os, glob, re
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
import configs
from plot import _on_white

params = configs.load_configs()
sweep_dir = params.base_path + "sweep_rRMSE/"

models = {
    "S-R Galerkin ROM": ("srpodgal_spectral_testing", "#6A3D9A"),
    "S-R OpInf ROM (optimal penalty weights)": ("sropinf_spectral_regularized_testing", "#FF3C00"),
}

def load_sweep(abbrev):
    """{amplitude(%): array of per-trajectory relative RMSE in %} from sweep_rRMSE/{abbrev}_amp*.txt."""
    out = {}
    for path in glob.glob(sweep_dir + f"{abbrev}_amp*.txt"):
        m = re.search(r"_amp([0-9.]+)\.txt$", path)
        if not m:
            continue
        vals = []
        with open(path) as f:
            for line in f:
                if line.startswith("#") or not line.strip():
                    continue
                vals.append(float(line.split()[1]))
        out[float(m.group(1))] = np.array(vals) * 100
    return out

model_data = {name: load_sweep(ab) for name, (ab, _) in models.items()}
all_amps = sorted(set().union(*[set(d) for d in model_data.values()]))
if not all_amps:
    print(f"[warn] no txt found in {sweep_dir} -- run the amplitude sweep first")

n_amp, n_models = len(all_amps), len(models)
xpos = np.arange(n_amp)                                   # categorical, evenly spaced groups
group_w = 0.66
box_w = group_w / n_models * 0.80                         # thin boxes
offsets = (np.arange(n_models) - (n_models - 1) / 2) * (group_w / n_models)

fig, ax = plt.subplots(figsize=(14, 6))
for gi in range(n_amp):                                   # alternating light shading bins each amplitude
    if gi % 2 == 0:
        ax.axvspan(gi - 0.5, gi + 0.5, color=_on_white("gray", 0.06), zorder=0)

for mi, (name, (abbrev, color)) in enumerate(models.items()):
    d = model_data[name]
    for gi, amp in enumerate(all_amps):
        if amp not in d:
            continue
        e = d[amp]
        surv = e[np.isfinite(e)]
        ndiv = int(np.sum(~np.isfinite(e)))
        pos = xpos[gi] + offsets[mi]
        if surv.size:
            bp = ax.boxplot([surv], positions=[pos], widths=box_w, patch_artist=True,
                            medianprops=dict(color="black", lw=1.5),
                            flierprops=dict(marker="o", markersize=3, markeredgecolor=_on_white("black", 0.5)))
            for patch in bp["boxes"]:
                patch.set_facecolor(_on_white(color, 0.6))
            if ndiv:
                ax.text(pos, surv.max(), f"{ndiv} div", ha="center", va="bottom", fontsize=9, color=color)
        elif ndiv:
            ax.text(pos, 7, "all\ndiv", ha="center", va="bottom", fontsize=9, color=color)

finite_max = max((d[a][np.isfinite(d[a])].max() for d in model_data.values()
                  for a in d if np.isfinite(d[a]).any()), default=150.0)
ax.set_yscale("log")
finite_min = min((d[a][np.isfinite(d[a])].min() for d in model_data.values()
                  for a in d if np.isfinite(d[a]).any()), default=1.0)
ax.set_ylim(finite_min * 0.7, finite_max * 1.5)
_lo = int(np.ceil(np.log10(finite_min * 0.7))); _hi = int(np.floor(np.log10(finite_max * 1.5)))
_ticks = [10.0 ** k for k in range(_lo, _hi + 1)]
ax.set_yticks(_ticks)
ax.set_yticklabels([f"{t:g}%" for t in _ticks])
ax.set_xticks(xpos)
ax.set_xticklabels([f"{a:g}" for a in all_amps])
ax.set_xlim(-0.5, n_amp - 0.5)
# ax.set_xlabel("perturbation amplitude (% of base norm)", fontsize=20)
# ax.set_ylabel("testing relative RMSE (%)", fontsize=20)
ax.tick_params(labelsize=16)
ax.grid(True, axis="y", which="major", color=_on_white("#b0b0b0", 0.25))
handles = [Patch(facecolor=_on_white(c, 0.6), label=n) for n, (_, c) in models.items()]
ax.legend(handles=handles, fontsize=14, loc="upper left", framealpha=1.0)

plt.tight_layout()
_fig_dir = params.base_path + "figures/sweep/"
os.makedirs(_fig_dir, exist_ok=True)
plt.savefig(_fig_dir + "fig5_perturbation_boxplot.png", dpi=300)
plt.savefig(_fig_dir + "fig5_perturbation_boxplot.eps", dpi=300)
plt.show()
print("saved Fig 5 ->", _fig_dir + "fig5_perturbation_boxplot.png / .eps")
